# LangChain: Agents & Tools

## Outline
1. مفهوم Agent و ReAct loop
2. ساخت tool با `@tool`
3. ساخت Agent با tool های خودمان
4. Streaming خروجی agent
5. Tool error handling با middleware
6. Built-in tools: Wikipedia (انگلیسی و فارسی) و DuckDuckGo
7. ابزارهای رایگان کاربردی دیگر: ArXiv، Open-Meteo، Yahoo Finance
8. جمع‌بندی و منابع

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)


## ۱. مفهوم Agent

Agent = LLM + Tools + Loop

**ReAct pattern:**
1. LLM **فکر** می‌کنه (reasoning)
2. **تصمیم** می‌گیره کدوم tool استفاده کنه
3. Tool رو **اجرا** می‌کنه
4. نتیجه رو **می‌بینه** و دوباره فکر می‌کنه
5. تا رسیدن به جواب نهایی ادامه می‌ده

قدیمی: `initialize_agent` + `AgentType`  
جدید: `create_agent`

## ۲. ساخت Tool با `@tool`

با دکوریتور `@tool` هر تابع پایتون به یک tool تبدیل می‌شه. مدل از روی **نام تابع**، **type hint ها** و مهم‌تر از همه **docstring** تصمیم می‌گیره کی و چطور tool رو صدا بزنه؛ پس docstring رو دقیق بنویسید.

In [5]:
from datetime import date

@tool
def get_today_date(text: str) -> str:
    """Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string."""
    return str(date.today())

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input should be a valid Python math expression.
    Example: '2 + 2', '15 * 4', '100 / 5'"""
    # توجه: eval حتی با builtins خالی کاملاً امن نیست؛ برای محیط واقعی از یک parser ریاضی استفاده کنید.
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """Get weather for a city.

    IMPORTANT:
    Always pass city name in English (e.g. London, Tehran, Tokyo).
    Never use Persian or translated names.
    """
    # داده‌ی ساختگی برای آموزش؛ در واقعیت باید به API وصل بشید (بخش ۷ رو ببینید)
    weather_data = {
        "Tehran": "25°C, Sunny",
        "London": "15°C, Cloudy",
        "New York": "20°C, Partly cloudy",
        "Tokyo": "28°C, Humid",
    }
    return weather_data.get(city, f"Weather data for {city} not available.")


# مشاهده metadata که مدل می‌بینه
print(f"Tool name: {get_today_date.name}")
print(f"Description: {get_today_date.description}")
print(f"Args: {get_today_date.args}")


Tool name: get_today_date
Description: Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string.
Args: {'text': {'title': 'Text', 'type': 'string'}}


## ۳. ساخت Agent با tool های خودمان

In [7]:
agent = create_agent(
    model=llm,
    tools=[get_today_date, calculate, get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "What is 25% of 300?"}]
})
print(response["messages"][-1].content)


25% of 300 is 75.


سؤال فارسی هم جواب می‌گیره: مدل اسم شهر رو طبق docstring به انگلیسی (`Tehran`) به tool می‌ده و جواب رو دوباره فارسی می‌نویسه.

In [11]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "هوای تهران چند درجه است؟"}]
})
print(response["messages"][-1].content)


هوای تهران در حال حاضر ۲۵ درجه سانتی‌گراد و آفتابی است.


## ۴. Streaming خروجی Agent

با `stream_mode="values"` بعد از هر step وضعیت کامل پیام‌ها رو می‌گیرید و می‌تونید مراحل ReAct رو ببینید: درخواست کاربر ← فراخوانی tool ← نتیجه‌ی tool ← جواب نهایی.

In [13]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage

print("=== Agent Streaming ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "دمای تهران چند درجه است؟"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if isinstance(latest, HumanMessage):
        print(f"[User]: {latest.content}")
    elif isinstance(latest, AIMessage):
        if latest.tool_calls:
            for tc in latest.tool_calls:
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
        elif latest.content:
            print(f"[AI]: {latest.content}")
    elif isinstance(latest, ToolMessage):
        print(f"[Tool Result]: {latest.content[:100]}")


=== Agent Streaming ===
[User]: دمای تهران چند درجه است؟
[Tool Call]: get_weather({'city': 'Tehran'})
[Tool Result]: 25°C, Sunny
[AI]: دمای تهران ۲۵ درجه سانتی‌گراد و آفتابی است.


In [19]:
print("=== Agent Streaming ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "اگر دمای هوای کنونی تهران را x در نظر بگیریم 25 درصد x چه قدر است؟"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if isinstance(latest, HumanMessage):
        print(f"[User]: {latest.content}")
    elif isinstance(latest, AIMessage):
        if latest.tool_calls:
            for tc in latest.tool_calls:
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
        elif latest.content:
            print(f"[AI]: {latest.content}")
    elif isinstance(latest, ToolMessage):
        print(f"[Tool Result]: {latest.content[:100]}")


=== Agent Streaming ===
[User]: اگر دمای هوای کنونی تهران را x در نظر بگیریم 25 درصد x چه قدر است؟
[Tool Call]: get_weather({'city': 'Tehran'})
[Tool Result]: 25°C, Sunny
[Tool Call]: calculate({'expression': '0.25 * 25'})
[Tool Result]: 6.25
[AI]: دمای هوای کنونی تهران 25 درجه سانتی‌گراد است. 25 درصد از این دما برابر با 6.25 درجه سانتی‌گراد است.


> نکته: اگه به جای `isinstance` از `hasattr(latest, 'name')` استفاده کنید، پیام کاربر (`HumanMessage`) هم به اشتباه به عنوان `Tool Result` چاپ می‌شه. برای همین نوع پیام رو دقیق با `isinstance` چک می‌کنیم.

## ۵. Tool Error Handling با Middleware

اگه tool خطا بده، بدون مدیریت خطا کل agent می‌شکنه. با `@wrap_tool_call` خطا رو می‌گیریم و به شکل یک `ToolMessage` به مدل برمی‌گردونیم تا خودش راه دیگه‌ای امتحان کنه یا توضیح بده.

In [23]:
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Middleware برای مدیریت خطاهای tool"""
    print("handle_tool_errors is called.")
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: {str(e)}. Please try a different approach.",
            tool_call_id=request.tool_call["id"]
        )

@tool
def risky_tool(number_a: str, number_b: str) -> str:
    """A tool that might fail. Input: two numbers. number_a / number_b"""
    return str(int(number_a) / int(number_b))  # اگه number_b صفر باشه: ZeroDivisionError

agent_safe = create_agent(
    model=llm,
    tools=[risky_tool, calculate],
    middleware=[handle_tool_errors],
    system_prompt=(
        "You are a math assistant. "
        "ALWAYS use the risky_tool for ANY division operation, even if you think the result is undefined. "
        "Never answer math questions directly — always call the appropriate tool first."
    )
)
response = agent_safe.invoke({
    "messages": [{"role": "user", "content": "What is 10 divided by 0?"}]
})
print(response["messages"][-1].content)


handle_tool_errors is called.
It seems that the division operation resulted in an error due to division by zero. If you have any other questions or need assistance with a different calculation, feel free to ask!


## ۶. Built-in Tools

تا اینجا tool ها رو خودمون نوشتیم. LangChain برای خیلی از سرویس‌ها tool آماده داره. لیست کامل: [Tools integrations](https://docs.langchain.com/oss/python/integrations/tools).

### ۶.۱ Wikipedia (انگلیسی)
```
pip install wikipedia langchain-community
```

In [ ]:
import wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ✅ تنظیم user-agent — بدون این، API ویکی‌پدیا ممکنه پاسخ خالی برگردونه
wikipedia.set_user_agent("LangChain-Course-Bot/1.0 (educational purposes)")

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,           # تعداد مقاله‌ها
        doc_content_chars_max=1000,  # حداکثر طول متن هر مقاله
    )
)

agent_wiki = create_agent(
    model=llm,
    tools=[wiki_tool, calculate, get_today_date],
    system_prompt="You are a research assistant. Use Wikipedia for factual questions.",
)

question = "Who invented Python programming language?"
response = agent_wiki.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)


### ۶.۲ Wikipedia (فارسی)

برای جستجو در **ویکی‌پدیای فارسی** کافیه پارامتر `lang="fa"` رو به `WikipediaAPIWrapper` بدیم. system prompt رو هم فارسی می‌نویسیم تا مدل جواب رو فارسی و بر اساس نتیجه‌ی ویکی‌پدیا بده.

نکته: ویکی‌پدیای فارسی از انگلیسی کوچک‌تره؛ اگه نتیجه‌ی خوبی نگرفتید، همون سؤال رو با `lang="en"` امتحان کنید.

In [ ]:
wiki_tool_fa = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        lang="fa",                 # ویکی‌پدیای فارسی
        top_k_results=2,
        doc_content_chars_max=1500,
    )
)

agent_wiki_fa = create_agent(
    model=llm,
    tools=[wiki_tool_fa],
    system_prompt=(
        "تو یک دستیار پژوهشی هستی. برای سؤال‌های دانشی حتماً از ابزار ویکی‌پدیا استفاده کن، "
        "عبارت جستجو را به فارسی بنویس و پاسخ نهایی را کوتاه و به فارسی بده."
    ),
)

question = "زبان برنامه‌نویسی پایتون را چه کسی ساخته است؟"
response = agent_wiki_fa.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)


### ۶.۳ DuckDuckGo (جستجوی وب، بدون API key)
```
pip install ddgs
```
برای اطلاعات به‌روز که در دانش مدل نیست.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()

agent_ddg = create_agent(
    model=llm,
    tools=[ddg_search, calculate, get_today_date],
    system_prompt=(
        "You are a web search assistant. "
        "Use DuckDuckGo to find up-to-date information."
    ),
)

question = "Latest LangChain version 2026"
response = agent_ddg.invoke({"messages": [{"role": "user", "content": question}]})
print("=== DuckDuckGo Agent ===")
print(response["messages"][-1].content)


## ۷. ابزارهای رایگان کاربردی دیگر

| ابزار | کاربرد | نیاز به API key | نصب |
|---|---|---|---|
| Wikipedia | دانش عمومی | خیر | `pip install wikipedia` |
| DuckDuckGo | جستجوی وب | خیر | `pip install ddgs` |
| ArXiv | مقالات علمی | خیر | `pip install arxiv` |
| Open-Meteo | آب‌وهوای واقعی | خیر | `pip install requests` |
| Yahoo Finance News | اخبار سهام | خیر | `pip install yfinance` |

### ۷.۱ ArXiv — جستجوی مقالات علمی
```
pip install arxiv
```
> ابزار آماده‌ی `ArxivQueryRun` در `langchain_community` با `arxiv` نسخه‌ی ۳ خطای `'Search' object has no attribute 'results'` می‌ده (چون متد `Search.results()` حذف شده). برای همین اینجا خودمون با `@tool` و `arxiv.Client().results(search)` یک tool می‌سازیم که با همه‌ی نسخه‌ها کار می‌کنه.

مناسب برای دستیار پژوهشی: عنوان، نویسنده و خلاصه‌ی مقاله‌ها رو برمی‌گردونه.

In [5]:
import arxiv

@tool
def search_arxiv(topic: str, max_results: int = 2) -> str:
    """Search ArXiv for scientific papers on a topic.
    Always write the topic in English. Returns title, authors, date, link and summary of each paper."""
    client = arxiv.Client()
    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    papers = []
    for p in client.results(search):
        papers.append(
            f"Title: {p.title}\n"
            f"Authors: {', '.join(a.name for a in p.authors)}\n"
            f"Published: {p.published.date()}\n"
            f"URL: {p.pdf_url}\n"
            f"Summary: {p.summary[:800]}"
        )
    return "\n\n".join(papers) or "No papers found."

agent_arxiv = create_agent(
    model=llm,
    tools=[search_arxiv],
    system_prompt=(
        "You are a research assistant. Use the ArXiv tool to find papers. "
        "Answer in Persian and mention the paper titles."
    ),
)

question = "دو مقاله درباره‌ی ReAct prompting برای agent ها پیدا کن و خلاصه کن."
response = agent_arxiv.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)


دو مقاله درباره‌ی "ReAct prompting" برای عامل‌ها به شرح زیر است:

1. **عنوان:** GEM: Graph-Enhanced Mixture-of-Experts with ReAct Agents for Dialogue State Tracking  
   **نویسندگان:** Ziqi Zhu, Adithya Suresh, Tomal Deb, Iman Abbasnejad  
   **تاریخ انتشار:** 2026-05-06  
   **لینک:** [مقاله](https://arxiv.org/pdf/2605.04449v1)  
   **خلاصه:** این مقاله به بررسی ردیابی وضعیت گفتگو (DST) می‌پردازد که نیاز به استخراج دقیق اطلاعات ساختاری از مکالمات چند دامنه دارد. نویسندگان یک چارچوب جدید به نام GEM را معرفی می‌کنند که مدل‌های زبانی و درک گفتگوی ساختاریافته را با استدلال مبتنی بر عامل‌های ReAct ترکیب می‌کند. این رویکرد به طور دینامیک بین متخصصان مختلف جابجا می‌شود و برای وظایف پیچیده تولید ارزش، عامل‌های ReAct را ادغام می‌کند که استدلال ساختاریافته‌ای بر اساس زمینه گفتگو انجام می‌دهند.

2. **عنوان:** On the road to percent accuracy IV: ReACT -- computing the non-linear power spectrum beyond $Λ$CDM  
   **نویسندگان:** Benjamin Bose, Matteo Cataneo, Tilman Tröster, Qianli Xia, Catherine H

## ۸. جمع‌بندی

- Agent = LLM + Tools + Loop (الگوی ReAct)
- با `@tool` هر تابعی رو tool کنید؛ **docstring** مهم‌ترین بخشه.
- با `stream` مراحل فکر کردن و فراخوانی tool رو ببینید.
- با middleware خطای tool ها رو مدیریت کنید.
- قبل از نوشتن tool جدید، لیست tool های آماده رو نگاه کنید.

## منابع
* لیست کامل tool ها و integration ها: [LangChain Tools Integrations](https://docs.langchain.com/oss/python/integrations/tools)